## Agent Loop 设计

---

> 本笔记承接 04_MCP 工具接入设计，聚焦「复杂任务多步推理分支」的完整落地方案：
> **agent_loop 路由 → 决策 / 执行循环 → 四道防线防卡死 → 错误治理（重试 / 降级 / 兜底）
> → 统一友好收尾 → 全链路可观测**

### 背景

智能问答已具备 direct（闲聊/常识）、rag（知识库）、tool（单次工具调用）三个分支，
但真实用户问题常是**多步复合任务**：「帮我规划明天从珠海去广州：先查天气，
再查公交路线」。这类任务需要模型拆解为子任务、逐个调工具、观察结果、
反思下一步——这就是 Agent Loop（复杂推理分支）的职责。

### 一、整体架构（循环模型）

```
┌─────────────────────────────────────────────────────────────┐
│ ① 意图层（super_router）                                    │
│   识别为多步复合任务 → route: agent_loop                    │
│   关键词：先…然后… / 对比 / 基于以上结果                    │
└───────────────────────────┬─────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│ ② 初始化（agent_loop_entry）                                │
│   loop_trace = [] · loop_step = 0 · loop_started_at = now   │
└───────────────────────────┬─────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│ ③ 决策层（loop_decide · LLM）                               │
│   四道防线检查：①步数上限 ②整轮超时 ③死循环 ④单步超时      │
│   输出 JSON{done, reasoning, tool, args}                    │
└──────────────┬──────────────────────────────┬───────────────┘
               │ done（完成）                 │ 继续（还有子任务）
               ▼                              ▼
┌───────────────────────────┐   ┌─────────────────────────────┐
│ ④ 收尾层（generate/audit）│   │ ⑤ 执行层（loop_execute）    │
│   基于全部轨迹汇总回答     │   │   参数校验 → 人工确认(high) │
│   审计落库（可追溯）       │   │   超时保护 → 自动重试 → 降级│
└───────────────────────────┘   │   结果/错误写 loop_trace    │
                                └─────────────┬───────────────┘
                                              │ 回到 ③（循环，最多 MAX_STEPS）
                                              ▼
                                        （防卡死兜底出口）
```

**核心设计原则**：终止出口只有一条——`loop_decide` 判定 `loop_done=true`
→ `generate` 汇总 → `audit` 落库。完成 / 超步数 / 超时 / 死循环四种路径
全部走同一出口，保证"不会无限循环、卡死必兜底"，轨迹全程可审计。

### 二、完整链路（一次复杂任务的路径）

| 步骤 | 发生什么 | 关键点 |
| --- | --- | --- |
| 1. 路由 | super_router 识别多步任务 → route: agent_loop | 关键词：先…然后… / 对比 |
| 2. 初始化 | entry 清空轨迹、记录起始时间 | 起始时间用于整轮熔断 |
| 3. 决策 | LLM 先列子任务清单 → 输出 JSON{done, tool, args} | 四道防线前置检查 |
| 4. 执行 | 校验参数 → 高风险人工确认 → 调工具 | 每次执行独立超时保护 |
| 5. 回环 | 结果写入轨迹 → 再次决策 | 直到 done=true |
| 6. 收尾 | generate 基于全部轨迹汇总 → audit 落库 | 轨迹即"为什么这么答"证据 |

**四种终止路径（防卡死矩阵）**

| 终止路径 | 触发条件 | 用户看到什么 |
| --- | --- | --- |
| 正常完成 | 全部子任务执行完，done=true | 汇总回答 |
| 步数上限 | loop_step >= MAX_STEPS(6) | 「已尝试 N 步仍未完成，先基于已有结果回答」 |
| 整轮超时 | 总耗时 > 180s | 「处理超时，先返回已完成的中间结果」 |
| 死循环 | 同一工具+同参数连续 >=2 次 | 「检测到重复执行同一操作，停止继续尝试」 |

### 三、循环控制与错误治理

#### 1. 四道防线（防卡死）

| 防线 | 参数 | 检查位置 | 兜底语义 |
| --- | --- | --- | --- |
| ① 步数上限 | MAX_STEPS=6 | loop_decide 前置 | 超步数直接 done |
| ② 整轮熔断 | LOOP_TOTAL_TIMEOUT=180s | loop_decide 前置 | 超时直接 done |
| ③ 死循环检测 | 同工具+同参数连续>=2 | loop_decide 前置 | 语义重复直接 done |
| ④ 单步超时 | STEP_LLM_TIMEOUT=40s / TOOL_TIMEOUT=30s | 决策/执行内 | 超时记 error，走治理链 |

#### 2. 错误治理链（工具失败怎么办）

```
工具失败
    │
    ├─ 可重试错误（超时/连接/限流/5xx）── 自动重试 1 次（TOOL_RETRY）
    │        │
    │        └─ 仍失败 ── TOOL_FALLBACK 备选工具降级（配置化映射）
    │                    │
    │                    └─ 仍失败 ── 记 error 轨迹 ── 模型看到后换工具/换参数
    │                                                 │
    └─ 业务错误（参数校验/权限）── 不重试，直接记 error 交模型换策略
                                                      │
                                            全失败 → 友好兜底（_terminate）
```

**可配置扩展点**：`TOOL_FALLBACK = {"amap_maps_weather": ["web_search"], ...}`
（同功能双源时启用）；参数集中在文件顶部常量区，后续可挪 yaml/数据库热更。

### 四、关键代码

#### 1. 四道防线参数（`backend/app/agent/loop.py`）

```python
MAX_STEPS = 6             # 最大推理步数（防死循环，第一道防线）
STEP_LLM_TIMEOUT = 40     # 单步 LLM 决策整体超时（模型链底层各档 15/20s，此为整链兜底）
TOOL_TIMEOUT = 30         # 单次工具执行超时（秒）
TOOL_RETRY = 1            # 工具失败自动重试次数（仅可重试错误）
LOOP_TOTAL_TIMEOUT = 180  # 整轮循环总时长上限（秒），超出即熔断走兜底

TOOL_FALLBACK: dict[str, list[str]] = {
    # "amap_maps_weather": ["web_search"],   # 同功能双源时启用
}

_RETRYABLE_MARKERS = (
    "timeout", "timed out", "connection", "connect",
    "refused", "closed", "503", "502", "500", "429", "rate",
    "超时", "连接",
)
```

#### 2. 死循环检测（`_detect_repeat`）

```python
def _detect_repeat(trace: list) -> str | None:
    '''同一工具 + 同一参数连续出现 >=2 次 → 返回终止原因（否则 None）。
    覆盖"模型反复调用同一工具同一参数"的原地打转场景。'''
    last_key = None
    count = 0
    for t in trace:
        key = (t.get("tool"),
               json.dumps(t.get("args") or {}, sort_keys=True, ensure_ascii=False))
        if key == last_key:
            count += 1
        else:
            last_key, count = key, 1
        if count >= 2:
            return f"检测到连续 {count} 次重复调用工具 {t.get('tool')}（参数相同）"
    return None
```

#### 3. 单步超时包装（`_call_with_timeout`）

```python
def _call_with_timeout(fn, timeout: float, default_error: str) -> dict:
    '''把任意同步调用包上超时（线程池 + future.result(timeout)）。
    超时后线程仍在后台运行（同步世界无法强杀），但主流程立即返回超时错误，
    由循环控制接管（重试 / 降级 / 兜底），不会阻塞整轮推理。'''
    pool = ThreadPoolExecutor(max_workers=1)
    fut = pool.submit(fn)
    try:
        return fut.result(timeout=timeout)
    except FutureTimeout:
        return {"success": False, "result": None, "error": default_error}
    except Exception as e:
        return {"success": False, "result": None,
                "error": f"{default_error}: {str(e)[:120]}"}
    finally:
        pool.shutdown(wait=False)  # 关键：不等待后台线程，避免超时后仍被阻塞
```

#### 4. 决策节点防线检查（`loop_decide` 前置）

```python
def loop_decide(state: AgentState) -> AgentState:
    writer = _get_writer()
    trace = state.get("loop_trace") or []
    step = state.get("loop_step") or 0

    # ① 步数上限
    if step >= MAX_STEPS:
        return _terminate(writer, f"达到步数上限 {MAX_STEPS}",
                          f"已尝试 {step} 步仍未完成全部子任务，先基于已有结果回答")
    # ③ 死循环检测
    repeat = _detect_repeat(trace)
    if repeat:
        return _terminate(writer, repeat, repeat + "，先基于已有结果回答")
    # ② 整轮总时长熔断
    started = state.get("loop_started_at") or time.time()
    if time.time() - started > LOOP_TOTAL_TIMEOUT:
        return _terminate(writer, f"整轮处理超时（>{LOOP_TOTAL_TIMEOUT}s）",
                          f"处理超时，先返回已完成的中间结果")
    # ④ 单步 LLM 整体超时（_call_with_timeout 包装 chat_complete）
    ...
```

#### 5. 执行节点错误治理（`loop_execute` 核心段）

```python
    # 执行：超时保护 + 自动重试 + 备选工具降级
    r = _execute_one(tool, args, ctx, TOOL_TIMEOUT)
    if not r["success"] and _is_retryable(r["error"]):
        # 自动重试（最多 TOOL_RETRY 次；已确认过的 high 工具不重复确认）
        for attempt in range(TOOL_RETRY):
            time.sleep(0.5)  # 短暂退避
            r = _execute_one(tool, args, ctx, TOOL_TIMEOUT)
            if r["success"]:
                entry["retried"] = True
                break
        # 备选工具降级
        if not r["success"]:
            for fb in TOOL_FALLBACK.get(tool, []):
                if not registry.get(fb):
                    continue
                r = _execute_one(fb, args, ctx, TOOL_TIMEOUT)
                entry["fallback_to"] = fb
                if r["success"]:
                    break
```

#### 6. 统一兜底出口（`_terminate`）

```python
def _terminate(writer, reason: str, friendly: str) -> dict:
    '''统一兜底出口：推送友好提示 + 结束循环（三种终止路径复用）'''
    logger.info(f"agent_loop 终止: {reason}")
    _push(writer, {"type": "thinking", "content": friendly})
    return {"loop_done": True, "loop_reasoning": reason}
```

> 另：`_get_writer()` 安全包装 `get_stream_writer`——非 LangGraph 运行上下文
> （脚本/单测直调节点）时返回 no-op，避免图外调用直接抛
> `Called get_config outside of a runnable context`。

### 五、实战一：复杂任务实测（真实 MCP 工具池）

**问题**：「帮我规划明天从珠海站去广州南站的行程：先查一下珠海今天的天气，
再查从珠海站到广州南站的公交路线」

**真实执行轨迹**（26 工具池 = 9 内置 + 15 高德 + 2 演示，qwen-max 决策，21.2s）：

| step | 决策工具 | 结果 | 说明 |
| --- | --- | --- | --- |
| 1 | amap_maps_weather | 珠海市 晴 25°C | 子任务 1 完成 |
| 2 | amap_maps_weather | 珠海市 晴（重复） | 参数不同未触发死循环 |
| 3 | amap_maps_direction_transit_integrated | API 失败：INVALID_PARAMS | 直接填地名，缺坐标 |
| 4 | amap_maps_geo | 广东省/珠海市 坐标解析 | 规则 6 生效：先地理编码 |
| 5 | 决策 done=true | 汇总回答 | 正常收尾 |

**关键结论**：模型自动完成「天气 → 公交 → 失败换 geo」的拆解与反思；
决策 prompt 规则 6（坐标类参数必须先调地理检索工具）让失败链路自动修正。

### 六、实战二：死循环检测与自动重试（真实触发）

#### 场景 A：模型反复调用同一工具同参数（web_search 空结果）

```
step1 web_search("珠海今天天气") -> 空结果
step2 web_search("珠海今天天气") -> 空结果   ← 模型没换策略
step3 决策前 _detect_repeat 命中 → 终止
```

结果：第 4 步被拦下，不再白白耗光 6 步 → 「检测到连续 2 次重复调用工具
web_search（参数相同），停止继续尝试，先基于已有结果回答」。

#### 场景 B：工具瞬时故障自动重试（单测验证）

```python
# 首次抛 Connection closed，第二次成功
def fail_once(args, ctx):
    calls["n"] += 1
    if calls["n"] == 1:
        raise RuntimeError("Connection closed by remote")
    return "ok-after-retry"

# loop_execute 结果：
#   {step:1, tool:"verify_fail_once", retried:True, result:"ok-after-retry"}
#   （错误匹配可重试标记 → 自动重试 1 次成功）
```

#### 场景 C：主工具全失败 → 备选工具降级（单测验证）

```python
# TOOL_FALLBACK = {"verify_main": ["verify_fb"]}
# verify_main 连续失败 -> 自动切 verify_fb 成功
# 轨迹：{fallback_to:"verify_fb", result:"fallback-result"}
```

### 七、验证结果（单测 + 端到端）

| 验证项 | 方式 | 结果 |
| --- | --- | --- |
| 死循环检测（同工具同参数） | 单测 3 例 | 命中/不命中判定正确 |
| 可重试错误判定 | 单测 4 例 | 超时/连接/限流重试，业务错误不重试 |
| 单步超时熔断 | sleep 工具 + 1s 超时 | 1s 返回超时错误，主流程不阻塞 |
| 自动重试 | fail_once 工具 | 重试成功，retried=True |
| 备选降级 | TOOL_FALLBACK 映射 | 主失败 → 备选成功，fallback_to 标记 |
| 复杂任务拆解 | 真实 MCP 工具池 26 工具 | 天气→公交→geo→done，21.2s |
| 死循环真实触发 | web_search 空结果反复 | 第 4 步拦截，友好兜底 |
| 回归（改动前行为） | 两步推理等 15 项 | 15/15 PASS，无破坏 |

**一句话总结**：Agent Loop = 决策/执行回环（复杂任务拆解）+ 四道防线
（步数 / 整轮超时 / 死循环 / 单步超时）+ 错误治理链（重试 → 降级 → 兜底）
+ 统一友好收尾——模型拆不出就兜底，卡死必被拦，全程轨迹可审计。